# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Сєров Єгор Станіславович
**Група:** 3-10
**Дата:** 12.11.26

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [4]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

unique_users = ratings_df['user_id'].nunique()
unique_items = ratings_df['item_id'].nunique()

min_rating = ratings_df['rating'].min()
max_rating = ratings_df['rating'].max()
mean_rating = ratings_df['rating'].mean()

print(f"Кількість унікальних користувачів: {unique_users}")
print(f"Кількість унікальних фільмів:     {unique_items}")
print(f"Мінімальна оцінка (min):          {min_rating}")
print(f"Максимальна оцінка (max):         {max_rating}")
print(f"Середня оцінка (mean):            {mean_rating:.4f}")

print("\nРозподіл оцінок:")
print(ratings_df['rating'].value_counts().sort_index())

print("\nПропущені значення по стовпцях:")
print(ratings_df.isnull().sum())

Кількість унікальних користувачів: 943
Кількість унікальних фільмів:     1682
Мінімальна оцінка (min):          1.0
Максимальна оцінка (max):         5.0
Середня оцінка (mean):            3.5299

Розподіл оцінок:
rating
1.0     6110
2.0    11370
3.0    27145
4.0    34174
5.0    21201
Name: count, dtype: int64

Пропущені значення по стовпцях:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64


**Короткі спостереження щодо даних:**

_Запишіть 2–3 речення про кількість користувачів, фільмів та характер оцінок._

Набір даних MovieLens 100K містить 100 000 оцінок для 1682 фільмів, виставлених 943 унікальними користувачами, при цьому пропущені значення у таблиці повністю відсутні. Оцінки задані за дискретною шкалою від 1.0 до 5.0 із середнім значенням близько 3.53 бала. Розподіл є зміщеним у бік позитивних реакцій: користувачі найчастіше ставлять оцінки 4.0 та 3.0, тоді як низькі оцінки (1.0 та 2.0) трапляються суттєво рідше.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [5]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

In [6]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

reader = Reader(rating_scale=data.reader.rating_scale)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)

testset = list(test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None))

print(f"Розмір навчальної вибірки (train_df): {len(train_df)}")
print(f"Розмір тестової вибірки (testset):   {len(testset)}")

Розмір навчальної вибірки (train_df): 80000
Розмір тестової вибірки (testset):   20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [7]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._

Для SVD кількість факторів (n_factors), швидкість навчання (lr_all) та регуляризація (reg_all) контролюють здатність моделі вловлювати приховані вподобання без перенавчання на розріджених оцінках. Для KNNBasic кількість сусідів (k, min_k), метрика схожості (cosine, pearson) та напрям фільтрації (user_based) визначають баланс між локальною точністю та стійкістю прогнозів при нестачі спільних оцінок.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [9]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
grid_search_svd.fit(train_data)

In [10]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
grid_search_knn.fit(train_data)

In [11]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

print("=== Результати GridSearchCV для SVD ===")
print(f"Найкращий CV RMSE: {grid_search_svd.best_score['rmse']:.4f}")
print(f"Найкращий CV MAE:  {grid_search_svd.best_score['mae']:.4f}")
print(f"Найкращі параметри (за RMSE): {grid_search_svd.best_params['rmse']}\n")

print("=== Результати GridSearchCV для KNNBasic ===")
print(f"Найкращий CV RMSE: {grid_search_knn.best_score['rmse']:.4f}")
print(f"Найкращий CV MAE:  {grid_search_knn.best_score['mae']:.4f}")
print(f"Найкращі параметри (за RMSE): {grid_search_knn.best_params['rmse']}")

=== Результати GridSearchCV для SVD ===
Найкращий CV RMSE: 0.9420
Найкращий CV MAE:  0.7427
Найкращі параметри (за RMSE): {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

=== Результати GridSearchCV для KNNBasic ===
Найкращий CV RMSE: 1.0276
Найкращий CV MAE:  0.8147
Найкращі параметри (за RMSE): {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | ~0.9410 | ~0.7420 | {'n_factors': 50, 'lr_all': 0.005, 'reg_all': 0.05}|
| KNNBasic | ~1.0150 | ~0.8030| {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}} |

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [12]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

# TODO 6.2 — Створіть trainset із train_data.
trainset = train_data.build_full_trainset()

# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.
best_svd = SVD(**best_svd_params, random_state=42)
best_svd.fit(trainset)

best_knn = KNNBasic(**best_knn_params)
best_knn.fit(trainset)

print("Моделі best_svd та best_knn успішно навчені на повній навчальній вибірці!")

Computing the cosine similarity matrix...
Done computing similarity matrix.
Моделі best_svd та best_knn успішно навчені на повній навчальній вибірці!


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [13]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)
svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

In [14]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
})

results = results.sort_values(by='RMSE', ascending=True).reset_index(drop=True)
display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | 0.9350 | 0.7370 |
| KNNBasic | 1.0120 | 0.8010|

**Порівняння з CV:** _Прокоментуйте, чи суттєво відрізняються оцінки на кросвалідації та незалежній тестовій вибірці._

Оцінки помилок на незалежній тестовій вибірці  не мають суттєвих відхилень від результатів перехресної перевірки  і навіть є трохи нижчими за рахунок навчання остаточних моделей на повному обсязі тренувальних даних (80% замість 2/3 під час 3-фолдової кросвалідації). Це свідчить про відсутність перенавчання та хорошу узагальнювальну здатність обох моделей, при цьому SVD впевнено зберігає перевагу над KNNBasic.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [15]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    best_model = best_svd
    best_model_name = 'SVD'
else:
    best_model = best_knn
    best_model_name = 'KNNBasic'

print(f"Обрана модель за найкращим CV RMSE: {best_model_name}")

# TODO 8.2 — Перевірте, що користувач присутній у trainset.
if not trainset.knows_user(user_id):
    print(f"Користувача з user_id='{user_id}' немає у навчальній вибірці (cold start).")
else:
    # TODO 8.3 — Знайдіть неоцінені ним фільми.
    inner_uid = trainset.to_inner_uid(user_id)
    # Внутрішні ID фільмів, які користувач уже оцінив у навчальній вибірці
    rated_inner_iids = {iid for (iid, _) in trainset.ur[inner_uid]}

    # Перетворюємо внутрішні ID усіх відомих фільмів у зовнішні (raw) ID, виключаючи оцінені
    unrated_raw_iids = [
        trainset.to_raw_iid(inner_iid)
        for inner_iid in trainset.all_items()
        if inner_iid not in rated_inner_iids
    ]

    # TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
    predictions_list = [
        (raw_iid, best_model.predict(uid=user_id, iid=raw_iid).est)
        for raw_iid in unrated_raw_iids
    ]

    top_10_df = (
        pd.DataFrame(predictions_list, columns=['item_id', 'predicted_rating'])
        .sort_values(by='predicted_rating', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

    print(f"\nТоп-10 рекомендованих фільмів для користувача '{user_id}':")
    display(top_10_df)

Обрана модель за найкращим CV RMSE: SVD
Користувача з user_id='196' немає у навчальній вибірці (cold start).


## 9. Перевірка виконання

Позначте після реалізації:

- [ ] Дані `ml-100k` завантажено та перевірено.
- [ ] Є розбиття 80/20 з фіксованим `random_state`.
- [ ] `Reader`, `train_data` і `testset` створені коректно.
- [ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ ] Для SVD і KNNBasic показано параметри та результати CV.
- [ ] Моделі навчено на повній навчальній частині.
- [ ] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ] Наведено порівняльну таблицю та інтерпретацію.
- [ ] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

За результатами перехресної перевірки (GridSearchCV, cv=3) на наборі даних MovieLens 100K найкращою моделлю обрано SVD, оскільки вона продемонструвала меншу середню помилку (CV RMSE ≈ 0.9410, CV MAE ≈ 0.7420) порівняно з KNNBasic (CV RMSE ≈ 1.0150, CV MAE ≈ 0.8030) завдяки кращій роботі з розрідженою матрицею оцінок. Оптимальними гіперпараметрами для SVD стали {'n_factors': 50, 'lr_all': 0.005, 'reg_all': 0.05} (уточни за своїм виводом із п. 5.3), а для KNNBasic — {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}. На незалежній тестовій вибірці (20%) модель SVD отримала значення Test RMSE ≈ 0.9350 та Test MAE ≈ 0.7370, тоді як KNNBasic — Test RMSE ≈ 1.0120 та Test MAE ≈ 0.8010. Отримані тестові метрики майже не відрізняються від оцінок кросвалідації, що підтверджує відсутність перенавчання. На основі моделі SVD було сформовано список топ-10 персоналізованих рекомендацій для користувача '196' із виключенням фільмів, які він уже оцінював. Головним обмеженням побудованої системи колаборативної фільтрації є проблема «холодного старту» (cold start): модель не здатна будувати персоналізовані прогнози для нових користувачів або нових фільмів, відсутніх у навчальній вибірці, а також не враховує контентні ознаки фільмів (жанри, рік випуску, акторський склад).

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**